In [1]:
from pathlib import Path
import pandas as pd

# === Pfade anpassen ===
BASE_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "data"
OUT_DIR  = BASE_DIR / "merged_groups"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === Gruppen & Model Codes ===
GROUPS = {
    "Nike_SM": ["HJ9198", "FD2722", "DH2987", "DJ6258", "DN3577"],
    "Nike_SF": ["DD1873", "DH3158", "FB2208", "FZ1689", "HM6804"],
    "Nike_CM": ["HQ2592", "HV8154", "DR7882", "FD1146", "DD1391"],
    "Nike_CF": ["DM0113", "IM1652", "DV1238", "HQ7540", "HQ2593"],
}

def read_csv_robust(path: Path) -> pd.DataFrame:
    # robust gegen Encoding-Probleme
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc, engine="python")
        except Exception:
            pass
    # wenn alles fehlschlägt, einmal "normal" werfen lassen (zeigt den echten Fehler)
    return pd.read_csv(path, engine="python")

def merge_group(group_name: str, codes: list[str]) -> pd.DataFrame:
    frames = []
    missing = []

    for code in codes:
        file_path = BASE_DIR / f"nike_reviews_{code}_EN.csv"
        if not file_path.exists():
            missing.append(str(file_path))
            continue

        df = read_csv_robust(file_path)

        # Metadaten, damit du später sauber filtern kannst
        df["model_code"] = code
        df["nike_group"] = group_name
        df["brand"] = "Nike"
        df["sustainability"] = "Sustainable" if group_name.startswith("Nike_S") else "Conventional"
        df["gender"] = "Male" if group_name.endswith("M") else "Female"

        frames.append(df)

    if missing:
        print(f"\n⚠️ Fehlende Dateien in {group_name}:")
        for m in missing:
            print(" -", m)

    if not frames:
        raise ValueError(f"Keine Dateien für {group_name} eingelesen (alle fehlen oder Fehler beim Lesen).")

    merged = pd.concat(frames, ignore_index=True, sort=False)

    # Optional: Duplikate entfernen, falls IDs mehrfach vorkommen
    if "id" in merged.columns:
        merged = merged.drop_duplicates(subset=["id"], keep="first")

    out_path = OUT_DIR / f"{group_name}_Total.csv"
    merged.to_csv(out_path, index=False)

    print(f"✅ {group_name}: gespeichert -> {out_path.name} | Rows: {len(merged)} | Cols: {merged.shape[1]}")
    return merged

# === Nur die 4 Gruppen bauen ===
group_dfs = {}
for group_name, codes in GROUPS.items():
    group_dfs[group_name] = merge_group(group_name, codes)

print("\nFertig. Erzeugte Dateien:")
for g in GROUPS:
    print(" -", (OUT_DIR / f"{g}_Total.csv").as_posix())


✅ Nike_SM: gespeichert -> Nike_SM_Total.csv | Rows: 1345 | Cols: 28
✅ Nike_SF: gespeichert -> Nike_SF_Total.csv | Rows: 1549 | Cols: 28
✅ Nike_CM: gespeichert -> Nike_CM_Total.csv | Rows: 1376 | Cols: 28
✅ Nike_CF: gespeichert -> Nike_CF_Total.csv | Rows: 1274 | Cols: 28

Fertig. Erzeugte Dateien:
 - /Users/raul/Desktop/Masterarbeit/Code/data/merged_groups/Nike_SM_Total.csv
 - /Users/raul/Desktop/Masterarbeit/Code/data/merged_groups/Nike_SF_Total.csv
 - /Users/raul/Desktop/Masterarbeit/Code/data/merged_groups/Nike_CM_Total.csv
 - /Users/raul/Desktop/Masterarbeit/Code/data/merged_groups/Nike_CF_Total.csv


In [2]:
from pathlib import Path
import pandas as pd

# Pfad zu deinem Data-Ordner (ohne Unterordner)
DATA_DIR = Path.home() / "Desktop" / "Masterarbeit" / "Code" / "data"

# Welche Datei bekommt welche Labels?
FILES = {
    "Nike_SM_Total.csv": ("Sustainable", "Male"),
    "Nike_SF_Total.csv": ("Sustainable", "Female"),
    "Nike_CM_Total.csv": ("Conventional", "Male"),
    "Nike_CF_Total.csv": ("Conventional", "Female"),
}

frames = []

for fname, (sust, gender) in FILES.items():
    path = DATA_DIR / fname
    if not path.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")

    df = pd.read_csv(path)

    # Spalten hinzufügen/überschreiben für klare Nachvollziehbarkeit
    df["Sustainability"] = sust
    df["Gender"] = gender

    frames.append(df)

# Alle vier Dateien zusammenführen (Spalten dürfen unterschiedlich sein -> Union aller Spalten)
nike_total = pd.concat(frames, ignore_index=True, sort=False)

# Optional: Duplikate entfernen, falls identische Reviews (über id) mehrfach vorkommen
if "id" in nike_total.columns:
    nike_total = nike_total.drop_duplicates(subset=["id"], keep="first")

out_path = DATA_DIR / "Nike_Total.csv"
nike_total.to_csv(out_path, index=False)

print(f"✅ Fertig: {out_path}")
print(f"Rows: {len(nike_total)} | Cols: {nike_total.shape[1]}")
print("Verteilung:")
print(nike_total.groupby(["Sustainability", "Gender"]).size())


✅ Fertig: /Users/raul/Desktop/Masterarbeit/Code/data/Nike_Total.csv
Rows: 5016 | Cols: 30
Verteilung:
Sustainability  Gender
Conventional    Female     746
                Male      1376
Sustainable     Female    1549
                Male      1345
dtype: int64
